# EXP-012: Val-gap diagnostic against a STALE (frozen) validation set (R9)

**Status: engineering validation / proof of concept**, same caveats as all prior
notebooks this session.

**Purpose (R9 of the revision roadmap, Perspective/Domain):** EXP-008's held-out
validation set tracked the CURRENT true `drift_frac` at every step -- meaning it had
oracle-like access to the live disturbance state, which a real deployed system would
not have (you cannot label a held-out set with the "current true concept" if you don't
already know a disturbance is happening and how far it has progressed). This notebook
builds a genuinely fair alternative: a **stale/frozen** validation set, generated ONCE
before training starts (`drift_frac` permanently fixed at its pre-disturbance value,
0.0), and reused for the entire run -- exactly what a real system would have available
without any special knowledge of drift.

**Prediction if EXP-008's success depended on the oracle access**: under DRIFT, a
STALE validation set (still reflecting the OLD task) would itself become "wrong"
once the true task moves -- so train loss and stale-val loss might now diverge
similarly to how train/live-val diverged for label noise, breaking the
drift-vs-label-noise discrimination. Under LABEL NOISE, the stale set's clean labels
match the (unchanged) true task just as well as the live oracle set did, so the
label-noise result should be UNAFFECTED by staleness.


In [1]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt

torch.set_num_threads(1)
print("torch", torch.__version__, "| numpy", np.__version__)


C:\Users\USER\anaconda3\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


torch 2.5.1+cpu | numpy 2.0.2


## 1. Toy task, disturbances, both validation-set variants

In [2]:
def make_model(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))

MEAN0_BASE = np.array([-1.0, -1.0])
MEAN1_BASE = np.array([1.0, 1.0])
BASE_NOISE_SCALE = 0.5

def gen_batch(rng, drift_frac=0.0, flip_prob=0.0, batch_size=16):
    frac = float(np.clip(drift_frac, 0.0, 1.0))
    mean0 = (1 - frac) * MEAN0_BASE + frac * MEAN1_BASE
    mean1 = (1 - frac) * MEAN1_BASE + frac * MEAN0_BASE
    y_true = rng.integers(0, 2, size=batch_size)
    means = np.where(y_true[:, None] == 0, mean0, mean1)
    x = means + rng.normal(scale=BASE_NOISE_SCALE, size=(batch_size, 2))
    flip_mask = rng.random(batch_size) < flip_prob
    y_observed = np.where(flip_mask, 1 - y_true, y_true)
    return torch.tensor(x, dtype=torch.float32), torch.tensor(y_observed, dtype=torch.float32)

N_STEPS = 400
DISTURBANCE_ONSET = 200
RAMP_LEN = 40
WARMUP = 50
GAP_WINDOW = 10
GRAD_NOISE_MAX = 8.0
LABEL_FLIP_PROB = 0.35
VAL_BATCH_SIZE = 64
GAP_Z_THRESHOLD = 2.0

def _ramp(step):
    if step < DISTURBANCE_ONSET:
        return 0.0
    return min(step - DISTURBANCE_ONSET, RAMP_LEN) / RAMP_LEN

def _instant(step):
    return 1.0 if step >= DISTURBANCE_ONSET else 0.0

def drift_forecastable(step):
    return (_ramp(step), 0.0, 0.0)

def drift_unforecastable(step):
    return (_instant(step), 0.0, 0.0)

def instability_forecastable(step):
    return (0.0, GRAD_NOISE_MAX * _ramp(step), 0.0)

def instability_unforecastable(step):
    return (0.0, GRAD_NOISE_MAX * _instant(step), 0.0)

def label_noise_forecastable(step):
    return (0.0, 0.0, LABEL_FLIP_PROB * _ramp(step))

def label_noise_unforecastable(step):
    return (0.0, 0.0, LABEL_FLIP_PROB * _instant(step))

CONDITIONS = {
    "drift_forecastable": drift_forecastable, "drift_unforecastable": drift_unforecastable,
    "instability_forecastable": instability_forecastable, "instability_unforecastable": instability_unforecastable,
    "label_noise_forecastable": label_noise_forecastable, "label_noise_unforecastable": label_noise_unforecastable,
}

def auto_select_intervention_valgap(train_window, heldout_window, baseline_gap_mean, baseline_gap_std,
                                     z_threshold=GAP_Z_THRESHOLD):
    current_gap = float(np.mean(train_window)) - float(np.mean(heldout_window))
    z = (current_gap - baseline_gap_mean) / (baseline_gap_std + 1e-6)
    return "reduce" if z > z_threshold else "increase"


## 2. Single training run, val_mode='oracle' (EXP-008, live drift_frac) or 'stale' (R9, frozen at 0.0)

In [3]:
def run_trial(seed, disturbance_fn, val_mode):
    assert val_mode in ("oracle", "stale")
    train_rng = np.random.default_rng(seed)
    val_rng = np.random.default_rng(seed + 1_000_000)
    model = make_model(seed)
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = nn.BCEWithLogitsLoss()

    train_loss_hist, heldout_loss_hist = [], []
    threshold = baseline_gap_mean = baseline_gap_std = None

    for step in range(N_STEPS):
        drift_frac, grad_noise_extra, flip_prob = disturbance_fn(step)
        xb, yb = gen_batch(train_rng, drift_frac=drift_frac, flip_prob=flip_prob)

        opt.zero_grad()
        hidden = model[1](model[0](xb))
        logits = model[2](hidden).squeeze(-1)
        loss = loss_fn(logits, yb)
        loss.backward()

        train_loss_t = float(loss.item())
        if grad_noise_extra > 0:
            with torch.no_grad():
                for p in model.parameters():
                    if p.grad is not None:
                        p.grad.add_(torch.randn_like(p.grad) * grad_noise_extra)

        # R9: val_mode='stale' freezes drift_frac at 0.0 (pre-disturbance) regardless
        # of the CURRENT true drift_frac -- no oracle access to the live disturbance
        # state, exactly what a real deployed validation set would have.
        val_drift_frac = drift_frac if val_mode == "oracle" else 0.0
        with torch.no_grad():
            xv, yv = gen_batch(val_rng, drift_frac=val_drift_frac, flip_prob=0.0, batch_size=VAL_BATCH_SIZE)
            v_hidden = model[1](model[0](xv))
            v_logits = model[2](v_hidden).squeeze(-1)
            heldout_loss_t = float(loss_fn(v_logits, yv).item())

        train_loss_hist.append(train_loss_t)
        heldout_loss_hist.append(heldout_loss_t)

        if step == WARMUP:
            warm_train = np.array(train_loss_hist[:WARMUP])
            warm_gap = np.array(train_loss_hist[:WARMUP]) - np.array(heldout_loss_hist[:WARMUP])
            threshold = float(warm_train.mean() + 3 * warm_train.std() + 1e-6)
            baseline_gap_mean = float(warm_gap.mean())
            baseline_gap_std = float(warm_gap.std()) + 1e-6

        if step >= WARMUP and threshold is not None and train_loss_t > threshold:
            train_window = train_loss_hist[max(0, step - GAP_WINDOW + 1):step + 1]
            heldout_window = heldout_loss_hist[max(0, step - GAP_WINDOW + 1):step + 1]
            diag_gap = auto_select_intervention_valgap(train_window, heldout_window, baseline_gap_mean, baseline_gap_std)
            gap_now = float(np.mean(train_window)) - float(np.mean(heldout_window))
            return {"trigger_step": step, "diag_gap": diag_gap, "gap_now": gap_now}

        opt.step()

    return {"trigger_step": None, "diag_gap": None, "gap_now": None}


## 3. Run all conditions x both val_modes x 10 seeds

In [4]:
SEEDS = list(range(10))
results = []
for val_mode in ("oracle", "stale"):
    for cond_name, fn in CONDITIONS.items():
        for seed in SEEDS:
            r = run_trial(seed, fn, val_mode)
            results.append({"val_mode": val_mode, "condition": cond_name, "seed": seed,
                             "trigger_step": r["trigger_step"], "diag_gap": r["diag_gap"], "gap_now": r["gap_now"]})

df = pd.DataFrame(results)
print(f"Total trials: {len(df)}")
df.head(10)


Total trials: 120


,val_mode,condition,seed,trigger_step,diag_gap,gap_now
0,oracle,drift_forecastable,0,216,reduce,0.113585
1,oracle,drift_forecastable,1,217,increase,-0.051745
2,oracle,drift_forecastable,2,215,reduce,0.066267
3,oracle,drift_forecastable,3,219,increase,0.056838
4,oracle,drift_forecastable,4,215,reduce,0.055595
5,oracle,drift_forecastable,5,217,increase,-0.038321
6,oracle,drift_forecastable,6,215,increase,0.018019
7,oracle,drift_forecastable,7,215,reduce,0.105270
8,oracle,drift_forecastable,8,213,increase,-0.011038
9,oracle,drift_forecastable,9,217,increase,-0.055391


## 4. Comparison: oracle vs stale validation set, by condition

In [5]:
print(f"{'condition':28s}{'val_mode':>10s}{'n_trig':>8s}{'gap->reduce%':>14s}{'mean_gap_now':>14s}")
rows = []
for cond in CONDITIONS:
    for val_mode in ("oracle", "stale"):
        sub = df[(df["condition"] == cond) & (df["val_mode"] == val_mode) & df["trigger_step"].notna()]
        n = len(sub)
        gap_reduce_pct = (sub["diag_gap"] == "reduce").mean() * 100 if n else float("nan")
        mean_gap = sub["gap_now"].mean() if n else float("nan")
        print(f"{cond:28s}{val_mode:>10s}{n:8d}{gap_reduce_pct:14.0f}{mean_gap:14.4f}")
        rows.append({"condition": cond, "val_mode": val_mode, "n_triggered": n,
                      "gap_reduce_pct": gap_reduce_pct, "mean_gap_now": mean_gap})
comparison = pd.DataFrame(rows)


condition                     val_mode  n_trig  gap->reduce%  mean_gap_now
drift_forecastable              oracle      10            40        0.0259
drift_forecastable               stale      10           100        0.3238
drift_unforecastable            oracle      10             0       -0.0189
drift_unforecastable             stale      10           100        0.4946
instability_forecastable        oracle      10            40        0.0251
instability_forecastable         stale      10            40        0.0251
instability_unforecastable      oracle      10            30        0.0301
instability_unforecastable       stale      10            30        0.0301
label_noise_forecastable        oracle      10           100        0.2708
label_noise_forecastable         stale      10           100        0.2708
label_noise_unforecastable      oracle      10           100        0.1637
label_noise_unforecastable       stale      10           100        0.1637


## 5. Interpretation

In [6]:
def get_pct(cond, val_mode):
    row = comparison[(comparison["condition"] == cond) & (comparison["val_mode"] == val_mode)]
    return float(row["gap_reduce_pct"].iloc[0])

label_conds = ["label_noise_forecastable", "label_noise_unforecastable"]
nonlabel_conds = [c for c in CONDITIONS if c not in label_conds]

label_tp_oracle = np.mean([get_pct(c, "oracle") for c in label_conds])
label_tp_stale = np.mean([get_pct(c, "stale") for c in label_conds])

fp_oracle = {c: get_pct(c, "oracle") for c in nonlabel_conds}
fp_stale = {c: get_pct(c, "stale") for c in nonlabel_conds}

drift_conds = ["drift_forecastable", "drift_unforecastable"]
drift_fp_oracle = np.mean([fp_oracle[c] for c in drift_conds])
drift_fp_stale = np.mean([fp_stale[c] for c in drift_conds])

lines = []
lines.append("R9 -- DOES THE VAL-GAP DIAGNOSTIC STILL WORK WITHOUT ORACLE ACCESS TO THE LIVE DRIFT STATE?")
lines.append("")
lines.append(f"Label-noise true-positive rate (mean 'reduce' % across both label-noise conditions):")
lines.append(f"  oracle (live drift_frac) = {label_tp_oracle:.0f}%   stale (frozen at 0.0) = {label_tp_stale:.0f}%")
lines.append("")
lines.append("Drift false-positive rate (the condition R9's prediction specifically targets --")
lines.append("a stale val set should itself look 'wrong' once the true task has moved under drift):")
lines.append(f"  oracle = {drift_fp_oracle:.0f}%   stale = {drift_fp_stale:.0f}%")
lines.append("")
lines.append("Full false-positive breakdown, all non-label-noise conditions:")
for c in nonlabel_conds:
    lines.append(f"  {c:28s} oracle={fp_oracle[c]:.0f}%  stale={fp_stale[c]:.0f}%")

tp_holds = abs(label_tp_stale - label_tp_oracle) < 20
drift_fp_worsens = (drift_fp_stale - drift_fp_oracle) > 15

verdict = "R9 RESULT: "
if tp_holds and not drift_fp_worsens:
    verdict += ("the val-gap diagnostic's sensitivity survives staleness (label-noise detection "
                "essentially unchanged), and drift false-positives do NOT meaningfully worsen either -- "
                "the diagnostic's earlier oracle-dependence caveat appears NOT to have been load-bearing "
                "for this specific result. This is a genuinely reassuring finding, though it should still "
                "be treated cautiously given the small (10-seed) sample.")
elif tp_holds and drift_fp_worsens:
    verdict += ("label-noise sensitivity survives staleness, BUT drift false-positives rise substantially "
                "with a stale validation set -- exactly the predicted failure mode: once the true task "
                "has moved under drift, a frozen 'clean' validation set is itself now measuring against "
                "the WRONG target, making drift look like label-type corruption too. This means the "
                "diagnostic's earlier oracle-dependence caveat WAS partially load-bearing -- the false-"
                "positive numbers reported for EXP-008/R1 (30-40% on drift) likely UNDERSTATE the "
                "false-positive rate a real, non-oracle deployment would see under drift specifically.")
else:
    verdict += ("the val-gap diagnostic's behavior changes substantially under a stale validation set in "
                "ways beyond the single drift-false-positive prediction -- see the full breakdown above "
                "for the specific pattern. This diagnostic's numbers should not be treated as robust to "
                "the oracle-vs-stale distinction without re-reading this table.")

lines.append("")
lines.append(verdict)
lines.append("")
lines.append(f"Same Stage A caveats as all prior notebooks this session (n={len(SEEDS)} seeds, "
              "no pre-registered effect sizes, not the frozen Stage D experiment).")
print(chr(10).join(lines))


R9 -- DOES THE VAL-GAP DIAGNOSTIC STILL WORK WITHOUT ORACLE ACCESS TO THE LIVE DRIFT STATE?

Label-noise true-positive rate (mean 'reduce' % across both label-noise conditions):
  oracle (live drift_frac) = 100%   stale (frozen at 0.0) = 100%

Drift false-positive rate (the condition R9's prediction specifically targets --
a stale val set should itself look 'wrong' once the true task has moved under drift):
  oracle = 20%   stale = 100%

Full false-positive breakdown, all non-label-noise conditions:
  drift_forecastable           oracle=40%  stale=100%
  drift_unforecastable         oracle=0%  stale=100%
  instability_forecastable     oracle=40%  stale=40%
  instability_unforecastable   oracle=30%  stale=30%

R9 RESULT: label-noise sensitivity survives staleness, BUT drift false-positives rise substantially with a stale validation set -- exactly the predicted failure mode: once the true task has moved under drift, a frozen 'clean' validation set is itself now measuring against the WRON